# Full-res preprocessing CACHE job (one-time)

Streams the REAL full-res data once and writes the compacted cache that Phase B
(U-Net) and Phase C (world model) both read:

- `data/preprocessing_data_full/` — climatology + scaling params (small)
- `data/full_cache/X_<var>.zarr`, `y_<var>.zarr` — fp16 + zstd chunks (~100–190 GB at ×1)

**Accelerator: NONE (CPU session)** — this is dask streaming + writes; a CPU session
doesn't touch your GPU quota (9 h/session cap still applies).

**RESUMABLE**: every variable is its own file. If the 9 h cap kills the session,
just re-open the notebook and re-run — completed files are skipped, only the
in-flight variable is redone. Repeat until the output ends in `CACHE_OK`.

**Sizes**: `--subsample 3` shrinks the cache to ~1/3 (Phase B only).
Default ×1 is contiguous daily — Phase C-ready.

## 1. Clone the repo + install deps (~2 min, skips on reruns)

In [ ]:
import os, subprocess, sys, pathlib
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
os.chdir(repo)
sys.path.insert(0, str(pathlib.Path(repo) / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
print("setup done")

## 2. Run the cache job (live stage output; re-run after any kill until CACHE_OK)

In [ ]:
# %run scripts/full_cache.py              # x1: Phase C-ready (~100-190 GB cache)
%run scripts/full_cache.py --subsample 1

## 3. Upload as PRIVATE Kaggle datasets (one-time, after CACHE_OK)

Split across datasets to stay under upload limits. For each: create a folder with
a `dataset-metadata.json` (title + id + CC-BY-4.0), copy the matching zarr files
into it, then create the dataset from this notebook:

```bash
mkdir -p /kaggle/working/ds_x && cp /kaggle/working/data/full_cache/X_*.zarr /kaggle/working/ds_x/ -r
cat > /kaggle/working/ds_x/dataset-metadata.json <<'JSON'
{"title": "climx-full-cache-X", "id": "nonewanitchollakit/climx-full-cache-x",
 "licenses": [{"name": "CC-BY-4.0"}]}
JSON
kaggle datasets create -p /kaggle/working/ds_x --private
# repeat for y_*.zarr (and more splits if a dataset hits the size cap)
```

Then on the competition Team page or the dataset's Share panel, add your teammates
as collaborators so everyone can attach the caches as notebook inputs.